Q2 Use the amazon review dataset: https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews/data for sentiment classification. Train BiLSTM model with 80% train, 10% validation and 10% test split to predict the sentiment as positive or negative. Use different drop out values (0.2, 0.4, 0.6) at lstm and embedding layers to compare the model performance in terms of training loss, validation macro F1 and test macro F1. 
Evaluate your model performance by introducing noise such as spelling mistake and synonym replacement in your test data.


In [42]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
import os
import re

from sklearn.metrics import f1_score

from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [2]:
Set_1 = pd.read_csv('test.csv', header=None)
Set_2 = pd.read_csv('train.csv', header=None)

DF = pd.concat([Set_1, Set_2])
DF.columns = ['Class', 'Title', 'Description']
DF

,Class,Title,Description
0,2,Great CD,My lovely Pat has one of the GREAT voices of h...
1,2,One of the best game music soundtracks - for a...,Despite the fact that I have only played a sma...
2,1,Batteries died within a year ...,I bought this charger in Jul 2003 and it worke...
3,2,"works fine, but Maha Energy is better",Check out Maha Energy's website. Their Powerex...
4,2,Great for the non-audiophile,Reviewed quite a bit of the combo players and ...
...,...,...,...
3599995,1,Don't do it!!,The high chair looks great when it first comes...
3599996,1,"Looks nice, low functionality",I have used this highchair for 2 kids now and ...
3599997,1,"compact, but hard to clean","We have a small house, and really wanted two o..."
3599998,1,what is it saying?,not sure what this book is supposed to be. It ...


In [3]:
DF_1 = DF[DF['Class'] == 1]
DF_2 = DF[DF['Class'] == 2]

DF = pd.concat([DF_1.iloc[:50000, :], DF_2.iloc[:50000, :]])
DF = DF.sample(frac = 1)

In [25]:
len(DF_1)

2000000

In [26]:
len(DF_2)

2000000

In [27]:
y = DF['Class'] - 1 # 1 is good, 0 is bad
X = DF.drop(columns = ['Class'])

In [28]:
X['Sentiment'] = X['Title'].astype(str) + " " + X['Description'].astype(str)
X.drop(columns = ['Title', 'Description'], inplace = True)

In [29]:
X = X[:10000]
y = y[:10000]

In [30]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text

In [31]:
X.iloc[:, 0] = X.iloc[:, 0].apply(clean_text)

In [32]:
def Vocabulary(texts, Max_len=20000):
    word_count = {}

    for text in texts:
        for word in text.split():
            if word in word_count:
                word_count[word] += 1
            else:
                word_count[word] = 1
    
    sorted_words = sorted(word_count.items(), key=lambda x: x[1], reverse=True)
    vocab = {"<pad>": 0, "<unk>": 1}

    for word, count in sorted_words[:Max_len - 2]:
        vocab[word] = len(vocab)

    return vocab

In [33]:
vocabulary = Vocabulary(list(X['Sentiment']))

In [34]:
def encode(text, vocab, Max_len=250):
    words = text.split()
    ids = []

    for word in words:
        if word in vocab:
            ids.append(vocab[word])
        else:
            ids.append(vocab["<unk>"])

    if len(ids) > Max_len:
        ids = ids[:Max_len]

    while len(ids) < Max_len:
        ids.append(vocab["<pad>"])

    return ids

In [35]:
def add_spelling_noise(text, prob=0.1):
    chars = list(text)
    for i in range(len(chars)):
        if random.random() < prob:
            chars[i] = ''
    return ''.join(chars)

synonyms = {
    "good": "nice",
    "bad": "poor",
    "great": "excellent",
    "worst": "terrible",
    "love": "like"
}

def synonym_replace(text):
    words = text.split()
    return " ".join([synonyms.get(w, w) for w in words])


In [36]:
Encoded_Dataset = []

for i in range(len(X)):
    encoded = encode(str(X.iloc[i, 0]), vocabulary, 250)
    Encoded_Dataset.append(encoded)

In [37]:
len(Encoded_Dataset), len(y)

(10000, 10000)

In [48]:
type(y)

pandas.core.series.Series

In [38]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, dropout=0.4):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # FIX: Explicit LSTM dropout
        self.lstm_dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        x = self.emb_dropout(x)

        _, (h_n, _) = self.lstm(x)

        h_forward = h_n[0]
        h_backward = h_n[1]

        h = torch.cat((h_forward, h_backward), dim=1)
        h = self.lstm_dropout(h)

        logits = self.fc(h)
        return logits.view(-1)

In [51]:
X_train, X_val, y_train, y_val = train_test_split(Encoded_Dataset, y, test_size=0.2, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_val, y_val, test_size=0.5, stratify=y_val)

In [49]:

# Get original text corresponding to test labels
_, X_temp_text, _, y_temp_text = train_test_split(X, y.to_numpy(),test_size=0.2,stratify=y,random_state=42)

X_test_text, _, y_test_text, _ = train_test_split(X_temp_text, y_temp_text,test_size=0.5,stratify=y_temp_text,random_state=42)

# Add noise
X_test_noisy_text = X_test_text.iloc[:, 0].apply(lambda text: synonym_replace(add_spelling_noise(str(text))))

# Encode noisy test set
Encoded_Noisy_Test = [encode(text, vocabulary, 250)for text in X_test_noisy_text]

X_test_noisy = torch.tensor(Encoded_Noisy_Test, dtype=torch.long)

test_noisy_ds = TensorDataset(X_test_noisy, y_test)
test_noisy_loader = DataLoader(test_noisy_ds, batch_size=32)

print("Clean and Noisy test loaders ready ✅")

Clean and Noisy test loaders ready ✅


In [52]:
# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.long)
X_val   = torch.tensor(X_val, dtype=torch.long)
X_test  = torch.tensor(X_test, dtype=torch.long)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_val   = torch.tensor(y_val.values, dtype=torch.float32)
y_test  = torch.tensor(y_test.values, dtype=torch.float32)

# Datasets
train_ds = TensorDataset(X_train, y_train)
val_ds   = TensorDataset(X_val, y_val)
test_ds  = TensorDataset(X_test, y_test)

# Loaders
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)


In [53]:
def evaluate(model, data_loader):
    device = next(model.parameters()).device
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            logits = model(x)

            preds = (torch.sigmoid(logits) > 0.5).int().cpu()

            all_preds.extend(preds)
            all_labels.extend(y.int())

    return f1_score(all_labels, all_preds, average="macro")


In [54]:
def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    epoch_check = {"train_loss": [], "val_f1": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device).view(-1)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        val_f1 = evaluate(model, val_loader)

        epoch_check["train_loss"].append(avg_train_loss)
        epoch_check["val_f1"].append(val_f1)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Macro F1: {val_f1:.4f}")

    return epoch_check

In [55]:
dropouts = [0.2, 0.4, 0.6]
results = {}

for d in dropouts:
    print(f"\nTraining Bi-LSTM with dropout: {d}")

    model = BiLSTM(vocab_size=len(vocabulary),embed_dim=100,hidden_dim=128,dropout=d)

    history = train_model(model, train_loader, val_loader, epochs=5)
    test_f1 = evaluate(model, test_loader)  # returns float
    noisy_test_f1 = evaluate(model, test_noisy_loader)
    
    print(f"Noisy Test Macro F1: {noisy_test_f1:.4f}\n")
    results[d] = {"train_loss": history["train_loss"][-1], "val_f1": history["val_f1"][-1],"test_f1": test_f1}


Training Bi-LSTM with dropout: 0.2
Epoch 1/5 | Train Loss: 0.5915 | Val Macro F1: 0.7454
Epoch 2/5 | Train Loss: 0.4539 | Val Macro F1: 0.8087
Epoch 3/5 | Train Loss: 0.3689 | Val Macro F1: 0.8137
Epoch 4/5 | Train Loss: 0.2985 | Val Macro F1: 0.8320
Epoch 5/5 | Train Loss: 0.2305 | Val Macro F1: 0.8210
Noisy Test Macro F1: 0.4478


Training Bi-LSTM with dropout: 0.4
Epoch 1/5 | Train Loss: 0.6100 | Val Macro F1: 0.7315
Epoch 2/5 | Train Loss: 0.5111 | Val Macro F1: 0.7650
Epoch 3/5 | Train Loss: 0.4495 | Val Macro F1: 0.7619
Epoch 4/5 | Train Loss: 0.3943 | Val Macro F1: 0.8028
Epoch 5/5 | Train Loss: 0.3507 | Val Macro F1: 0.7969
Noisy Test Macro F1: 0.4910


Training Bi-LSTM with dropout: 0.6
Epoch 1/5 | Train Loss: 0.6511 | Val Macro F1: 0.7015
Epoch 2/5 | Train Loss: 0.5771 | Val Macro F1: 0.7579
Epoch 3/5 | Train Loss: 0.5368 | Val Macro F1: 0.7732
Epoch 4/5 | Train Loss: 0.4973 | Val Macro F1: 0.7704
Epoch 5/5 | Train Loss: 0.4621 | Val Macro F1: 0.7362
Noisy Test Macro F1: 0.4

In [59]:
print(pd.DataFrame.from_dict(results))

                 0.2       0.4       0.6
train_loss  0.230485  0.350673  0.462078
val_f1      0.820991  0.796931  0.736196
test_f1     0.841994  0.776774  0.740267
